# КЕДР: проверка ответа коллеги

Коллега: вид `sbx_edm_dfip.v_detail_dmkb_dkb_dmsb` берёт только клиентов, у которых в КЕДР есть строка «Торговый эквайринг»; у 7841058379 такой строки нет. Этим же он объясняет низкий июль.

1. Колонка продукта в `Kedr.v_detail_*` и написание «Торговый эквайринг».
2. Три ИНН из письма за 202607: есть ли у них строка «Торговый эквайринг».
3. Июль 2026: совпадает ли набор ИНН вида с клиентами озера, у которых есть «Торговый эквайринг».
4. Низкий июль: суммы по месяцам у всех клиентов КЕДР и только у клиентов с эквайрингом.

Только чтение. Логины спрашиваются один раз.

In [ ]:
from getpass import getpass

import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 250)
pd.set_option("display.float_format", "{:,.2f}".format)

DRP_VIEW = "sbx_edm_dfip.v_detail_dmkb_dkb_dmsb"
LAKE_VIEWS = {"dmkb": "Kedr.v_detail_dmkb", "dmsb": "Kedr.v_detail_dmsb", "dkb": "Kedr.v_detail_dkb"}
CHECK_MONTH = 202607
MONTH_FROM, MONTH_TO = 202601, 202609
SAMPLE_INN = ["7841058379", "1208001968", "664301504099"]
ACQ_LIKE = "%эквайринг%"

if "drp" not in globals():
    drp = connect(
        to="DRP",
        user_params={"user_name": input("DRP login: ").strip(), "password": getpass("DRP password: ")},
    )

if "imp" not in globals():
    imp = connect(
        to="IMPALA",
        extra_options={"db": "sandbox_ai"},
        driver_args={"tez.queue.name": "ai"},
        kerberos={
            "keytab_path": "/home/jovyan/test_requests/tech.keytab",
            "use_credentials": True,
            "update_keytab": True,
        },
        user_params={"user_name": input("Login (keytab): ").strip()},
    )


def drp_fetch(sql):
    with drp:
        return drp.fetch(sql)


def imp_fetch(sql):
    with imp:
        imp.execute("SET MEM_LIMIT=4g")
        return imp.fetch(sql)


def norm_inn(v):
    s = "".join(ch for ch in str(v) if ch.isdigit())
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s or None


for view in LAKE_VIEWS.values():
    try:
        with imp:
            imp.execute(f"REFRESH {view}")
    except Exception as exc:
        print(f"REFRESH {view}: не удалось ({str(exc)[:150]})")
print("OK")

## 1. Колонка продукта и написание «Торговый эквайринг»

In [ ]:
PROD_COL = {}
for src, view in LAKE_VIEWS.items():
    desc = imp_fetch(f"DESCRIBE {view}")
    cols = desc.iloc[:, 0].str.lower().tolist()
    PROD_COL[src] = next((c for c in ("prod_name", "product_name", "product") if c in cols), None)
    print(f"{view}: колонка продукта = {PROD_COL[src]}")

parts = []
for src, view in LAKE_VIEWS.items():
    col = PROD_COL[src]
    if not col:
        continue
    df = imp_fetch(f"""
        SELECT cast({col} AS string) AS prod_name, count(*) AS rows_cnt, count(DISTINCT inn) AS inn_cnt
        FROM {view}
        WHERE yearmm = {CHECK_MONTH} AND lower(cast({col} AS string)) LIKE '{ACQ_LIKE}'
        GROUP BY 1
    """)
    if df is not None and len(df):
        df["src"] = src
        parts.append(df)
print("Продукты со словом «эквайринг» за", CHECK_MONTH)
display(pd.concat(parts, ignore_index=True) if parts else "не найдено")

## 2. Три ИНН из письма

Если коллега прав: у них за июль ни в одной витрине нет строки «Торговый эквайринг», а ЧОД по другим продуктам есть.

In [ ]:
variants = sorted(set(SAMPLE_INN) | {x.lstrip("0") for x in SAMPLE_INN})
inn_list = ", ".join(f"'{x}'" for x in variants)
parts = []
for src, view in LAKE_VIEWS.items():
    col = PROD_COL[src] or "''"
    df = imp_fetch(f"""
        SELECT cast(inn AS string) AS inn, cast({col} AS string) AS prod_name,
               count(*) AS rows_cnt, sum(nbi_ssp) AS nbi
        FROM {view}
        WHERE yearmm = {CHECK_MONTH} AND cast(inn AS string) IN ({inn_list})
        GROUP BY 1, 2
    """)
    if df is not None and len(df):
        df["src"] = src
        parts.append(df)
sample = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=["inn", "prod_name", "rows_cnt", "nbi", "src"])
sample["inn"] = sample["inn"].map(norm_inn)
sample["is_acq"] = sample["prod_name"].fillna("").str.lower().str.contains("эквайринг")
display(sample.sort_values(["inn", "src", "prod_name"]))
display(
    sample.groupby("inn", as_index=False)
    .agg(nbi_total=("nbi", "sum"), has_acq_row=("is_acq", "max"), products=("prod_name", "nunique"))
)

## 3. Июль: совпадает ли вид с «клиентами озера, у которых есть Торговый эквайринг»

Если коллега прав: `view_only` и `lake_acq_only` около 0, а `lake_no_acq` — это и есть клиенты, которых вид отсекает.

In [ ]:
view_inn = drp_fetch(f"SELECT DISTINCT inn FROM {DRP_VIEW} WHERE yearmm = {CHECK_MONTH}")
view_set = {norm_inn(x) for x in view_inn["inn"]} - {None}

parts = []
for src, view in LAKE_VIEWS.items():
    col = PROD_COL[src]
    acq_expr = f"max(CASE WHEN lower(cast({col} AS string)) LIKE '{ACQ_LIKE}' THEN 1 ELSE 0 END)" if col else "0"
    df = imp_fetch(f"""
        SELECT cast(inn AS string) AS inn, sum(nbi_ssp) AS nbi, {acq_expr} AS has_acq
        FROM {view}
        WHERE yearmm = {CHECK_MONTH}
        GROUP BY 1
    """)
    if df is not None and len(df):
        df["src"] = src
        parts.append(df)
lake = pd.concat(parts, ignore_index=True)
lake["inn"] = lake["inn"].map(norm_inn)
lake["nbi"] = pd.to_numeric(lake["nbi"], errors="coerce").fillna(0)
lake["has_acq"] = pd.to_numeric(lake["has_acq"], errors="coerce").fillna(0).astype(int)
by_src = lake.groupby(["inn", "src"], as_index=False).agg(nbi=("nbi", "sum"), has_acq=("has_acq", "max"))
lake_inn = by_src.groupby("inn", as_index=False).agg(nbi=("nbi", "max"), has_acq=("has_acq", "max"))

acq_set = set(lake_inn.loc[lake_inn["has_acq"] == 1, "inn"])
no_acq_set = set(lake_inn.loc[lake_inn["has_acq"] == 0, "inn"])
display(pd.DataFrame([
    {"check": "ИНН в виде", "value": len(view_set)},
    {"check": "ИНН озера с «Торговый эквайринг»", "value": len(acq_set)},
    {"check": "совпали", "value": len(view_set & acq_set)},
    {"check": "view_only: в виде, но без эквайринга в озере", "value": len(view_set - acq_set)},
    {"check": "lake_acq_only: эквайринг в озере есть, а в виде нет", "value": len(acq_set - view_set)},
    {"check": "lake_no_acq: в озере без эквайринга (вид их отсекает)", "value": len(no_acq_set)},
    {"check": "их ЧОД, всего", "value": round(lake_inn.loc[lake_inn["has_acq"] == 0, "nbi"].sum(), 2)},
]))

print("Примеры view_only:", sorted(view_set - acq_set)[:10])
print("Примеры lake_acq_only:", sorted(acq_set - view_set)[:10])

## 4. Низкий июль: это фильтр или сам источник?

Суммы `nbi_ssp` по витринам и месяцам: у всех клиентов и только у клиентов с «Торговый эквайринг». Если июль низкий и у всех клиентов, фильтр ни при чём: так в самом КЕДР, вопрос к ДФИП.

In [ ]:
parts = []
for src, view in LAKE_VIEWS.items():
    col = PROD_COL[src]
    acq_expr = f"max(CASE WHEN lower(cast({col} AS string)) LIKE '{ACQ_LIKE}' THEN 1 ELSE 0 END)" if col else "0"
    df = imp_fetch(f"""
        SELECT yearmm,
               count(*) AS inn_all,
               sum(nbi) AS nbi_all,
               sum(has_acq) AS inn_acq,
               sum(CASE WHEN has_acq = 1 THEN nbi ELSE 0 END) AS nbi_acq_clients
        FROM (
            SELECT cast(yearmm AS bigint) AS yearmm, cast(inn AS string) AS inn,
                   sum(nbi_ssp) AS nbi, {acq_expr} AS has_acq
            FROM {view}
            WHERE yearmm BETWEEN {MONTH_FROM} AND {MONTH_TO}
            GROUP BY 1, 2
        ) x
        GROUP BY yearmm
    """)
    if df is not None and len(df):
        df["src"] = src
        parts.append(df)
months = pd.concat(parts, ignore_index=True)
for c in ("inn_all", "nbi_all", "inn_acq", "nbi_acq_clients"):
    months[c] = pd.to_numeric(months[c], errors="coerce")
months["yearmm"] = months["yearmm"].astype(int)

print("По витринам:")
display(months.pivot_table(index="yearmm", columns="src", values=["nbi_all", "nbi_acq_clients"], aggfunc="sum"))
print("Сумма трёх витрин (клиент в двух витринах посчитан дважды, это только для динамики):")
display(months.groupby("yearmm")[["inn_all", "nbi_all", "inn_acq", "nbi_acq_clients"]].sum())